In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# plt.rcParams['font.family'] = 'Malgun Gothic' # For Windows
plt.rcParams["font.family"] = "AppleGothic"   # Mac
%matplotlib inline

In [2]:
import pandas as pd

df = pd.read_csv("cleaned_airbnb.csv")
df.head()

,name,host_since,host_response_time,host_acceptance_rate,neighbourhood_cleansed,neighbourhood_group_cleansed,property_type,room_type,accommodates,bathrooms,...,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,price_log1p,availability_rate_30,availability_rate_60,availability_rate_90,availability_rate_365,room_capacity,legal_flag
0,Lg Rm in Historic Prospect Heights,2009-12-11,NaN,NaN,Prospect Heights,Brooklyn,Private room in rental unit,Private room,1,1.0,...,1,0,0.05,5.303305,0.900000,0.950000,0.966667,0.991781,3.0,Legal
1,"1 Bedroom & your own Bathroom, Elevator Apartment",2010-07-04,NaN,100%,East Harlem,Manhattan,Private room in condo,Private room,2,1.0,...,1,0,0.58,4.418841,0.000000,0.000000,0.000000,0.558904,3.0,Legal
2,Luxury Brownstone in Boerum Hill,2010-07-13,within a few hours,40%,Boerum Hill,Brooklyn,Private room in home,Private room,2,2.5,...,1,0,0.28,6.641182,1.000000,0.816667,0.733333,0.893151,12.5,Legal
3,Spectacular West Harlem Garden Apt,2010-07-14,within an hour,97%,Harlem,Manhattan,Entire home,Entire home/apt,2,1.0,...,0,0,1.36,4.941642,0.233333,0.300000,0.277778,0.068493,3.0,Legal
4,“Work-from-home” from OUR home.,2010-07-16,within an hour,100%,Williamsburg,Brooklyn,Private room in rental unit,Private room,2,1.0,...,2,0,1.54,4.875197,0.466667,0.366667,0.422222,0.104110,3.0,Legal


### 데이트타입 변환

### price -> float 변환

In [3]:
# price 타입 str -> float 으로 변환
df['price']=(df['price'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False).astype(float))
df['price'].dtype

dtype('float64')

### price -> log 변환

In [4]:
# price 로그변환
df["price_log1p"] = np.log1p(df["price"])  # log(1+Fare)
df["price_log1p"]

0        5.303305
1        4.418841
2        6.641182
3        4.941642
4        4.875197
           ...   
22303    4.290459
22304    4.077537
22305    5.703782
22306    5.303305
22307    4.077537
Name: price_log1p, Length: 22308, dtype: float64

## 3/5 목 전처리
위치, 호스트, 라이센스 컬럼 -> 주석만 달아두기

## 1️⃣ 위치 컬럼 전처리 - 주석설명만
# 위치 변수 중요성: - 숙소 가격 및 수요에 위치가 큰 영향을 미치는 변수
group_clean
- 뉴욕의 5개의 큰 행정구 (borough)
- 위치 정보를 가장 큰 단위로 표현하는 변수
- 범주 수가 적어 모델 안정성이 높음
- 지역별 가격/불법 분포 해석이 용이
- 별도 전처리 없이 사용 가능

clean
- borough 내부의 세부 지역(neighborhood)
- 총 221개 범주 존재
- 일부 지역은 데이터가 매우 적어 희소 범주 발생
- 
- 전처리 방법: 상위 10개의 지역만 보기 + 나머지는 other 로 묶기
- 
상위 10만 보는 근거:
- - 데이터 비율 기준으로 주요 지역만 유지
- 희소 범주 감소
- 모델 과적합 방지

In [5]:
df['neighbourhood_cleansed'].value_counts(dropna=False)
# length 카테고리가 221개 , 일부는 표본이 1-2개뿐
# 긴꼬리 구조
# 상위 (안정된표본수를 가진 것) 20~30 개의 지역만 유지
# 나머지는 other 로 묶기



neighbourhood_cleansed
Bedford-Stuyvesant        1545
Midtown                   1461
Upper East Side           1042
Hell's Kitchen            1036
Harlem                    1031
                          ... 
Country Club                 2
Fort Wadsworth               1
Willowbrook                  1
Chelsea, Staten Island       1
Oakwood                      1
Name: count, Length: 221, dtype: int64

In [6]:
df['neighbourhood_cleansed'].value_counts(dropna=False)

neighbourhood_cleansed
Bedford-Stuyvesant        1545
Midtown                   1461
Upper East Side           1042
Hell's Kitchen            1036
Harlem                    1031
                          ... 
Country Club                 2
Fort Wadsworth               1
Willowbrook                  1
Chelsea, Staten Island       1
Oakwood                      1
Name: count, Length: 221, dtype: int64

In [7]:
df['neighbourhood_group_cleansed'].value_counts(dropna=False)

neighbourhood_group_cleansed
Manhattan        10205
Brooklyn          7455
Queens            3420
Bronx              912
Staten Island      316
Name: count, dtype: int64

In [8]:
# 1. 상위 10개 neighborhood 추출
top10_neighborhoods = df['neighbourhood_cleansed'].value_counts().head(10).index

# 2. 새로운 컬럼 생성 (원본 컬럼 유지)
df['neighbourhood_top10'] = df['neighbourhood_cleansed'].apply(
    lambda x: x if x in top10_neighborhoods else 'Other'
)

In [9]:
df['neighbourhood_top10'].value_counts()

neighbourhood_top10
Other                 12485
Bedford-Stuyvesant     1545
Midtown                1461
Upper East Side        1042
Hell's Kitchen         1036
Harlem                 1031
Upper West Side         929
Williamsburg            853
Bushwick                758
Crown Heights           648
Chelsea                 520
Name: count, dtype: int64

# 호스트 정보 전처리 - 설명 주석만 달아두기
1️⃣host_since
- str 타입 형태 -> 실제 날짜 데이터
- 호스트 계정 가입 날짜이다. (숙소등록 x,매출연관x)
- 의미 없는 컬럼
- 
- 전처리방법
- datetime 변환
-
2️⃣'host_response_time'
- str 타입 형테
- nan 존재 (비율 상당 1/5)
-
- nan 제거인가? 대체인가?
- 0시간 응답이다? => 응답 데이터 없음. 
- 호스트가 메시지 받은적 없음/ 응답률 비공개 / 새호스트 기록부족    가능성
- 응답시간 데이터 없는데 매출은 찍힌다. -> 비공개 가능성
-
-전처리 방법
- 'no_reponse_data'로 대체

-이유
- 정보유지
- 데이터손실 최소화
- 모델 사용 가능
- 수요 신뢰도 변수로 사용 가능 -(예:  빠른응답->예약 증가 에 관계있음)
-
3️⃣'host_acceptance_rate'
- 예약 요청 수락률
- nan 존재 (3466)
- 
- 예약이 없는데 가격은 존재.
- 즉시예약일 경우 승인 요청이 없어서일 가능성
- 신규 숙소의 가능성
- 미기록
- 의미 있는 결측으로 판단한다.
-
- 전처리방법
-'no_request' (예약 요청없음)으로 대체
- 중앙값/평균으로 대체? -> 실제 의미 왜곡
- 


In [10]:
# 1️⃣'host_since'
df['host_since'].value_counts(dropna=False)

host_since
2016-12-16    1096
2022-02-25     313
2023-08-22     311
2017-12-11     229
2015-12-16     226
              ... 
2025-02-11       1
2020-07-26       1
2025-02-16       1
2025-02-23       1
2024-03-16       1
Name: count, Length: 4568, dtype: int64

In [11]:
# host_since를 datetime으로 변환, 원본 컬럼 유지
df['host_since_dt'] = pd.to_datetime(df['host_since'], errors='coerce')

In [12]:
df['host_since_dt'].dtype

dtype('<M8[us]')

In [13]:
# 2️⃣'host_response_time'
df['host_response_time'].value_counts(dropna=False)

host_response_time
within an hour        11645
NaN                    4393
within a few hours     3439
within a day           1957
a few days or more      874
Name: count, dtype: int64

In [14]:
df.loc[df['host_response_time'].isna(), 'price']

0        200.0
1         82.0
5        139.0
14       171.0
16        62.0
         ...  
22298    571.0
22299    571.0
22300    571.0
22301    221.0
22303     72.0
Name: price, Length: 4393, dtype: float64

In [15]:
df['host_response_time'] = df['host_response_time'].fillna('no_reponse_data')

In [16]:
# 3️⃣'host_acceptance_rate' 예약 요청 수락률
df['host_acceptance_rate'].value_counts(dropna=False)

host_acceptance_rate
100%    4909
NaN     3466
97%     1407
99%     1006
0%       999
        ... 
26%        4
28%        3
6%         2
9%         2
3%         1
Name: count, Length: 97, dtype: int64

In [17]:
df.loc[df['host_acceptance_rate'].isna(), 'price']

0        200.0
5        139.0
11       216.0
12       165.0
14       171.0
         ...  
22298    571.0
22299    571.0
22300    571.0
22301    221.0
22303     72.0
Name: price, Length: 3466, dtype: float64

In [18]:
df['host_acceptance_rate']=df['host_acceptance_rate'].fillna('no_request')

## 기타 전처리 - 주석으로 달아두기

license
- 숙소등록/허가번호
- NaN 확인(최대치)
-
- 등록 NaN => 불법인가?
- 결론 아니다.
- 등록이 필요없는 숙소 (호텔와 같은) 는 등록 없어도 합법
- 
- 애초에 등록 대상이 아닌 숙소 
- 예
- private room / shared room / 30일 이상 장기임대 일 경우 nan이어도 정상
-
- 전처리방법
- 'unknown' (등록상태불명)으로 대체.
-
- 참고변수로 사용가능 

경우	설명
license 있음 + 규정 준수	   합법
license 없음 + 등록 대상	   불법
license 없음 + 등록 대상 아님	합법
license 있음 + 규정 위반	   불법 가능

호텔 -> 괜찮습니다.
규제: 단기임대    주거형+30박 미만 =규제적용
주거형+30박 이상  -> 단기임대가 아니니, 등록이 불필요하다.

In [19]:
df['license'].value_counts(dropna=False)

license
NaN                   17845
Exempt                 2064
OSE-STRREG-0000068       81
OSE-STRREG-0041458       24
OSE-STRREG-0001054       17
                      ...  
OSE-STRREG-0002916        1
OSE-STRREG-0002238        1
OSE-STRREG-0002922        1
OSE-STRREG-0002813        1
OSE-STRREG-0002894        1
Name: count, Length: 1895, dtype: int64

In [20]:
df['license']=df['license'].fillna('unknown')

In [21]:
#단기 임대 여부
df['is_short_term'] = df['minimum_nights'] < 30

In [22]:
#license 결측 여부
df['license_missing'] = df['license'].isna()

In [23]:
#호텔이냐 아니냐
df['is_hotel'] = df['property_type'].str.contains('hotel', case=False, na=False)

In [24]:
df['LL18_risk'] = (
    (df['is_short_term']) &
    (df['license_missing']) &
    (~df['is_hotel'])
    )
df['LL18_risk'].value_counts()

LL18_risk
False    22308
Name: count, dtype: int64

### vlaue_counts로 본 property_type의 종류를 거주용/숙박용 기준으로 나누어서 새 컬럼에 매핑.
- Residential   (주거형)
- Lodging       (숙박업형)

In [25]:
len(df['property_type'])

22308

In [26]:
property_map = {
    'Room in hotel': 'Lodging',
    'Room in boutique hotel': 'Lodging',

    'Entire rental unit': 'Residential',
    'Private room in rental unit': 'Residential',
    'Private room in home': 'Residential',
    'Entire home': 'Residential',
    'Entire condo': 'Residential',
    'Private room in townhouse': 'Residential',
    'Private room in condo': 'Residential',
    'Entire townhouse': 'Residential',
    'Entire loft': 'Residential',
    'Entire guest suite': 'Residential'
}

df['property_regulation_type'] = df['property_type'].map(property_map)

df['property_regulation_type'] = df['property_regulation_type'].fillna('Other')

In [27]:
df['property_regulation_type'].value_counts()

property_regulation_type
Residential    20335
Lodging         1026
Other            947
Name: count, dtype: int64

In [28]:
len(df['property_regulation_type'])

22308

In [29]:
df['legal_flag'] = 'Legal'  # 합법 컬럼 생성해주기.

# Residential + Entire + 단기(<30박) → 불법 가능
df.loc[
    (df['property_regulation_type'] == 'Residential') &
    (df['room_type'] == 'Entire home/apt') &
    (df['minimum_nights'] < 30),
    'legal_flag'
] = 'Illegal'

# 나머지는 Legal 그대로

In [30]:
df['legal_flag'].value_counts()

legal_flag
Legal      20808
Illegal     1500
Name: count, dtype: int64

In [31]:
# 최종 분석 데이터셋 확정하기
df_final = df[
    ( (df['property_regulation_type'] == 'Residential') & (df['legal_flag'] == 'Legal') )
    | ( df['property_regulation_type'] == 'Lodging' )
]

In [32]:
df.to_csv("cleaned_airbnb.csv", index=False)